<a href="https://colab.research.google.com/github/micah-shull/AI_Agents/blob/main/814_CBOv2_NodeTests.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This is a strong node-level suite — it’s doing what node tests *should* do: verify the “wiring contract” between nodes without needing LangGraph execution.

That said, there are **3 issues** I’d fix (two are likely test failures / brittleness), plus a few high-value adds.

---

## The big issues

### 1) `test_build_initial_state` asserts `config_snapshot is None`

```python
assert "config_snapshot" in state
assert state["config_snapshot"] is None
```

But your newer design is “config snapshot in state” (self-documenting runs). In practice you have two choices:

* **Option A (recommended):** `build_initial_state()` sets `config_snapshot` immediately (using `asdict(config)`), so it’s never None.
* **Option B:** only the load node sets it after config is available.

Right now, your test enforces **Option B**, but your other test (`test_load_data_node_success`) expects the node output includes config_snapshot. That part is fine.

Just be clear: if you ever move snapshotting to `build_initial_state`, this test will fail even though the system improved.

✅ Suggested change: make this assertion tolerant:

```python
assert "config_snapshot" in state
# may be None initially; load_data node should populate it
```

or remove the `is None` check entirely.

---

### 2) `test_load_data_node_success` expects fields your node may not return

You assert:

```python
assert out["config_snapshot"] is not None
assert out["run_timestamp"] is not None
assert "validation_warnings" in out
```

But the `make_load_data_node` code you shared earlier returned only:

```python
return {
  "customers": ...,
  "finance_signals": ...,
  "retail_signals": ...,
  "healthcare_signals": ...,
  "portfolio_rollup": ...,
  "errors": errors
}
```

So either:

* you updated the node to also return `config_snapshot`, `run_timestamp`, `validation_warnings` (great), **or**
* this test will fail.

✅ Fix guidance:

* Ensure the load node returns these keys explicitly if you want them in state.
* If you prefer “state mutation” style (returning only changed keys), then the test should only assert what the node returns.

Given your architecture, I’d *prefer* the load node returns:

* `validation_warnings`
* `config_snapshot`
* `run_timestamp`
  because they’re part of the “run record.”

---

### 3) `test_build_report_node` sets `run_start_time = 0.0` (brittle)

Your report builder computes runtime:

```python
processing_time = (time.time() - run_start) if run_start else None
```

If `run_start_time` is `0.0`, then:

* `if run_start` is falsey → runtime becomes None.

But your node returns `"processing_time"` anyway, and your test only asserts the key exists, not that it’s non-null. So it’ll pass.

However, it’s semantically weird. Better to set it to a real start time:

✅ Better:

```python
"run_start_time": time.time() - 0.01
```

Or simply:

```python
"run_start_time": 1.0
```

If you want deterministic tests, don’t use real `time.time()`—use `monkeypatch` to freeze time.

---

## Medium issues / improvements

### A) `ROOT` path insert likely wrong for real imports

You’re doing:

```python
ROOT = Path(__file__).resolve().parent
sys.path.insert(0, str(ROOT))
```

If `config.py` is at repo root and tests are also at repo root, fine.

But if tests live in `tests/`, then inserting the tests directory won’t necessarily allow importing `config`. Many repos instead insert the repo root.

✅ More robust pattern:

```python
ROOT = Path(__file__).resolve().parents[1]  # if tests/ folder
sys.path.insert(0, str(ROOT))
```

If your tests are in the root already, ignore this.

---

### B) `reports_dir` override in test may not be honored by `_ensure_reports_dir`

Your report util does:

```python
root = Path(__file__).resolve().parents[4]
reports_dir = root / config.reports_dir
```

If `config.reports_dir` is an absolute path (it is in your test), `root / abs` is fine (it resolves to abs). ✅

So this is okay.

---

## High-value tests to add (small, protect key behaviors)

### 1) Load node surfaces validation warnings (core trust feature)

Create customers but empty finance list, assert:

* `validation_warnings` contains “finance_signals is empty…”
* report includes “Validation Warnings” section when warnings exist (if you render it)

### 2) Classify node trigger threshold uses config

Right now your trigger threshold is config-driven (good). Add a test where:

* portfolio pct_positive_alignment = 0.09 → no trigger
* pct_positive_alignment = 0.10 → trigger fires

This locks in “no hidden thresholds.”

### 3) Segment view doesn’t miscount missing domains

Once you harden segment logic, add a test:

* finance domain_status = insufficient_data
* expect “2-domain” etc.

This is the one that prevents silent executive misreads.

---

## One quick correctness tweak for current tests

In `test_classify_patterns_node`, you assert:

```python
assert out["portfolio_rollup"]["customers_total"] == 1
```

This is fine, but it assumes your classify function uses that `max(len,1)` pattern.

If you ever change semantics (more honest 0), this will break. If you want the test to reflect truth rather than implementation hack, assert instead:

* key exists
* counts are integers
* pct fields are between 0 and 1

That makes the test robust to refactors.

---

## Suggested minimal edits to your existing tests (so they stay stable)

* Remove `config_snapshot is None` assertion in initial state test
* In load node success, assert only keys you *guarantee* the node returns (or ensure node returns the new metadata keys)
* Set `run_start_time` to a non-false value (or monkeypatch time)




In [ ]:
"""
Node tests for CBOv2 orchestrator.

Test each node with mock state; use closure nodes with config and tmp_path for report output.
Run from project root: pytest test_CBOv2_nodes.py -v --tb=short
"""
from __future__ import annotations

import json
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pytest
from config import CBOv2Config, CBOv2State
from agents.CBOv2.orchestrator.nodes import (
    build_initial_state,
    make_load_data_node,
    make_compute_trends_node,
    make_classify_patterns_node,
    make_build_report_node,
)


def _minimal_cbo_state() -> CBOv2State:
    return {
        "customers": [{"customer_id": "C1"}],
        "finance_signals": [{"customer_id": "C1", "balance_short": 100, "balance_long": 90, "missed_payments_short": 0, "missed_payments_long": 0}],
        "retail_signals": [{"customer_id": "C1", "spend_short": 110, "spend_long": 100, "frequency_short": 2, "frequency_long": 1}],
        "healthcare_signals": [{"customer_id": "C1", "utilization_short": 2, "utilization_long": 1, "gaps_short": 10, "gaps_long": 25}],
        "customer_trends": [],
        "portfolio_rollup": {},
        "errors": [],
        "run_start_time": None,
    }


# ----- build_initial_state -----

def test_build_initial_state():
    state = build_initial_state({"run_id": "test_run"})
    assert state["run_id"] == "test_run"
    assert state["goal"]["summary"]
    assert state["plan"]
    assert state["errors"] == []
    assert state["run_start_time"] is not None
    assert "config_snapshot" in state
    assert state["config_snapshot"] is None


# ----- load_data node -----

def test_load_data_node_success(tmp_path):
    (tmp_path / "customers.json").write_text(json.dumps([{"customer_id": "C1"}]))
    (tmp_path / "finance_signals.json").write_text(json.dumps([{"customer_id": "C1"}]))
    (tmp_path / "retail_signals.json").write_text(json.dumps([{"customer_id": "C1"}]))
    (tmp_path / "healthcare_signals.json").write_text(json.dumps([{"customer_id": "C1"}]))
    config = CBOv2Config(data_dir=str(tmp_path))
    node = make_load_data_node(config)
    state: CBOv2State = {"errors": []}
    out = node(state)
    assert "errors" in out
    assert len(out["errors"]) == 0
    assert out["customers"] == [{"customer_id": "C1"}]
    assert out["config_snapshot"] is not None
    assert out["run_timestamp"] is not None
    assert "validation_warnings" in out


def test_load_data_node_missing_file(tmp_path):
    (tmp_path / "customers.json").write_text("[]")
    config = CBOv2Config(data_dir=str(tmp_path))
    node = make_load_data_node(config)
    state: CBOv2State = {"errors": []}
    out = node(state)
    assert len(out["errors"]) > 0
    assert "load_data_failed" in out["errors"][0] or "not found" in out["errors"][0].lower()


# ----- compute_trends node -----

def test_compute_trends_node():
    config = CBOv2Config()
    node = make_compute_trends_node(config)
    state = _minimal_cbo_state()
    out = node(state)
    assert "customer_trends" in out
    assert len(out["customer_trends"]) == 1
    assert out["customer_trends"][0]["customer_id"] == "C1"
    assert "finance_trend" in out["customer_trends"][0]


# ----- classify_patterns node -----

def test_classify_patterns_node():
    config = CBOv2Config()
    # First get customer_trends from compute_trends
    state = _minimal_cbo_state()
    trends_node = make_compute_trends_node(config)
    state_with_trends = {**state, **trends_node(state)}
    # Then classify
    classify_node = make_classify_patterns_node(config)
    out = classify_node(state_with_trends)
    assert "portfolio_rollup" in out
    assert "top_risk_customers" in out
    assert "top_growth_customers" in out
    assert "executive_triggers" in out
    assert out["portfolio_rollup"]["customers_total"] == 1


# ----- build_report node -----

def test_build_report_node(tmp_path):
    config = CBOv2Config(reports_dir=str(tmp_path / "reports"))
    node = make_build_report_node(config)
    state: CBOv2State = {
        "portfolio_rollup": {"customers_total": 1, "customers_with_negative_alignment": 0, "customers_with_positive_alignment": 1},
        "executive_triggers": [{"name": "expansion_momentum", "verdict": "opportunity", "reason": "33%"}],
        "top_risk_customers": [],
        "top_growth_customers": [{"customer_id": "C1", "domains_positive": 2, "finance_trend": {"is_positive": True}, "retail_trend": {"is_positive": False}, "healthcare_trend": {"is_positive": True}}],
        "customer_trends": [{"customer_id": "C1", "domains_positive": 2, "domains_negative": 0}],
        "run_id": "test",
        "run_timestamp": "2026-03-02T12:00:00Z",
        "run_start_time": 0.0,
        "config_snapshot": {"version": "2.0"},
        "validation_warnings": [],
        "errors": [],
    }
    out = node(state)
    assert "report_markdown" in out
    assert "report_file_path" in out
    assert out["report_file_path"] is not None
    assert Path(out["report_file_path"]).exists()
    assert "processing_time" in out
    assert "CBOv2" in out["report_markdown"]
    assert "Cross-Business" in out["report_markdown"]
